In [ ]:
import requests
import json
import datetime
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

expected_units = {
    "temperature_2m": "°C",
    "cloud_cover": "%",
    "wind_speed_10m": "km/h"
}

def run_pipeline_for_city(city_name, lat, lon, engine):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&hourly=temperature_2m,wind_speed_10m,cloud_cover"

    try:
        response = requests.get(url)
    except requests.exceptions.RequestException as e:
        print(f"connection failed: {e}")
    if response.status_code == 200:
        print(f"connection succesfully : {response.status_code}")
        data = response.json()
    else:
        print(f"connection failed : {response.status_code}")
        return
    for param, unit in expected_units.items():
        actual_unit = data["hourly_units"][param]
        if actual_unit == unit:
            print(f"agreed: {param}: {unit}")
        else:
            print(f"faild: {param}: {unit}")

    now = datetime.datetime.now()
    now_str = now.strftime("%Y-%m-%d_%H-%M-%S")
    file_name = f"weather_hourly_{city_name}_{now_str}.json"

    with open(f"data/raw/{file_name}", "w") as f:
        json.dump(data, f)

    df = pd.DataFrame(data['hourly'])
    df['time'] = pd.to_datetime(df['time'])
    print("total null values : ", df['time'].isna().sum())
    print("duplicated values : ", df['time'].duplicated().sum())
    print(df["time"][df['temperature_2m'].idxmax()])

    df.to_sql(
    name=f"weather_{city_name}",
    con=engine,
    if_exists='replace',
    index=False
)
    
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(connection_string)


run_pipeline_for_city("luxor", 22.6872, 32.6396, engine)


connection succesfully : 200
agreed: temperature_2m: °C
agreed: cloud_cover: %
agreed: wind_speed_10m: km/h
total null values :  0
duplicated values :  0
2026-09-15 13:00:00
